# Hand-crafted transformer — building the algorithm from scratch

**Goal.** Build the weights of a 2-block transformer **explicitly** from the dlog-Fourier algorithm — no training. If full-dataset accuracy ≥ 95%, each mechanistic claim (V projections carry dlog Fourier of inputs; FFN1 neurons compute `cos(k·(la−lb))`; W_U reads Fourier into per-c logits) is verified as both necessary and sufficient.

**Standalone.** No external data required. Runs in under a minute on CPU. Produces the `handcrafted_state_dict.pt` that `../verify_handcrafted.py` loads.

**Algorithm encoded.** `c = a · b⁻¹ mod 97` ⇔ `lc = la − lb mod 96` (in dlog of generator g).

**Architecture.** d_model = 384, n_heads = 4 (head_dim = 96), 2 blocks (block 0 active, block 1 identity), SwiGLU FFN.

**d_model layout:**
- `0..95`   — la Fourier (cos+sin per k=1..48, packed as `[c1, s1, c2, s2, ...]`)
- `96..191` — lb Fourier (same packing)
- `192..239` — `cos(k·(la−lb))` channels, one dim per k
- `240..287` — `sin(k·(la−lb))` channels
- `288..379` — spare/zero
- `380..383` — position markers (one-hot)

**SwiGLU trick.** SwiGLU output = `W2 · (SiLU(W1·x) ⊙ (W3·x))`. We approximate the pure product `cos(k·la)·cos(k·lb)` by setting `W1` very small (so `SiLU(ε·x) ≈ ε·x/2`) and amplifying via `W2`. Limit `ε → 0` recovers exact product; with `ε = 0.01` the residual error is O(10⁻⁴).

**Outputs to `./outputs/handcrafted/`:**
- `handcrafted_state_dict.pt`
- `summary.json` — accuracy, per-stage activation diagnostics, Fourier-circle CV
- `figures.png` — W_U dlog circle + residual manifold + FFN1-mid neurons for k=11

In [ ]:
# Cell 1 — Setup
# If running in Colab and you want outputs on Drive, uncomment:
#   from google.colab import drive
#   drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, json, time, os, gc
from pathlib import Path
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

OUT_DIR = Path('./outputs/handcrafted')
OUT_DIR.mkdir(parents=True, exist_ok=True)

P = 97; PM1 = 96
D_MODEL = 384; N_HEADS = 4; HEAD_DIM = D_MODEL // N_HEADS  # 96
K_FREQS = list(range(1, PM1 // 2 + 1))  # k=1..48, all non-DC frequencies
K = len(K_FREQS)  # 48

# Layout in d_model
OFFSET_LA = 0
OFFSET_LB = 96
OFFSET_COS = 192  # cos((la-lb)) per k
OFFSET_SIN = 240  # sin((la-lb)) per k
OFFSET_POS = 380  # 4 position markers at 380..383

POS_MARKER_VAL = 5.0  # magnitude of position marker (tuned for RMSNorm survival)
EPS_SWIGLU = 0.01     # linearization scale for SiLU
GAIN_W2 = 1.0 / (EPS_SWIGLU * 0.5)  # 200 — undo SiLU(eps·x) ≈ eps·x/2

import matplotlib.pyplot as plt
BG, FG, GRID, MUTED = '#FAFAF7', '#2A2A2A', '#E5E3DD', '#6E6E6E'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'axes.edgecolor': FG, 'axes.linewidth': 0.6,
    'axes.labelcolor': FG, 'axes.titlecolor': FG,
    'xtick.color': FG, 'ytick.color': FG, 'text.color': FG,
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5, 'grid.alpha': 0.8,
})

# Discrete log setup
def is_gen(g, p):
    seen = set(); x = 1
    for _ in range(p-1):
        x = (x*g) % p
        if x in seen: return False
        seen.add(x)
    return len(seen) == p-1
for g in range(2, P):
    if is_gen(g, P): break
exp_table = np.zeros(PM1, dtype=np.int64); x = 1
for t in range(PM1): exp_table[t] = x; x = (x*g) % P
dlog = np.zeros(P, dtype=np.int64)
for t in range(PM1): dlog[exp_table[t]] = t
# Note: dlog[0] is undefined (0 is not in the multiplicative group). We never predict 0.

print(f'g={g}, K={K} frequencies, EPS={EPS_SWIGLU}, GAIN_W2={GAIN_W2}, OUT={OUT_DIR.resolve()}')

In [ ]:
# Cell 2 — Architecture (identical to trained models)
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__(); self.scale = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.scale

class GrokBlock(nn.Module):
    def __init__(self, d, nh, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(d); self.attn = nn.MultiheadAttention(d, nh, dropout=dropout, batch_first=True)
        self.norm2 = RMSNorm(d); self.w1 = nn.Linear(d, 4*d, bias=False)
        self.w2 = nn.Linear(4*d, d, bias=False); self.w3 = nn.Linear(d, 4*d, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h = self.norm1(x); o, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.dropout(o)
        h2 = self.norm2(x); gate = F.silu(self.w1(h2))
        return x + self.dropout(self.w2(gate * self.w3(h2)))

class GrokModel(nn.Module):
    def __init__(self, p=97, d=384, nh=4):
        super().__init__()
        self.tok_emb = nn.Embedding(p+2, d); self.pos_emb = nn.Embedding(4, d)
        self.blocks = nn.ModuleList([GrokBlock(d, nh, dropout=0.0) for _ in range(2)])
        self.norm_f = RMSNorm(d); self.head = nn.Linear(d, p, bias=False); self.p = p
    def forward(self, a, b):
        B = a.size(0)
        op = torch.full((B,), self.p, device=a.device, dtype=torch.long)
        eq = torch.full((B,), self.p+1, device=a.device, dtype=torch.long)
        tok = torch.stack([a, op, b, eq], dim=1)
        pos = torch.arange(4, device=a.device).unsqueeze(0).expand(B, -1)
        x = self.tok_emb(tok) + self.pos_emb(pos)
        for bl in self.blocks: x = bl(x)
        return self.head(self.norm_f(x)[:, -1, :])
    def forward_capture_pre_head(self, a, b):
        B = a.size(0)
        op = torch.full((B,), self.p, device=a.device, dtype=torch.long)
        eq = torch.full((B,), self.p+1, device=a.device, dtype=torch.long)
        tok = torch.stack([a, op, b, eq], dim=1)
        pos = torch.arange(4, device=a.device).unsqueeze(0).expand(B, -1)
        x = self.tok_emb(tok) + self.pos_emb(pos)
        for bl in self.blocks: x = bl(x)
        h = self.norm_f(x)[:, -1, :]
        return self.head(h), h
    def head_only(self, h):
        return self.head(h)

# Data
all_a, all_b, all_c = [], [], []
for a in range(1, P):  # skip a=0 (division undefined when la is taken later if a=0)
    for b in range(1, P):
        all_a.append(a); all_b.append(b); all_c.append((a * pow(b, P-2, P)) % P)
all_a_t = torch.tensor(all_a, device=device)
all_b_t = torch.tensor(all_b, device=device)
all_c_t = torch.tensor(all_c, device=device)
print(f'Data: {len(all_a)} (a,b) pairs')

m = GrokModel(P, D_MODEL, N_HEADS).to(device)
print(f'Model: d={D_MODEL}, n_heads={N_HEADS}, head_dim={HEAD_DIM}')
n_params = sum(p.numel() for p in m.parameters())
print(f'Params: {n_params/1e6:.2f}M')

In [ ]:
# Cell 3 — Build W_E and pos_emb
# W_E[t] = [Fourier of dlog(t), zeros..., zeros] for t ∈ {1,...,96}
# W_E[t] = 0 for t ∈ {0, op=97, eq=98}
# pos_emb[i, OFFSET_POS+i] = POS_MARKER_VAL for i ∈ {0..3}, else 0

W_E = np.zeros((P+2, D_MODEL), dtype=np.float32)
for t in range(1, P):  # t in {1..96}
    lt = dlog[t]  # dlog of token, in {0..95}
    for ki, k in enumerate(K_FREQS):
        theta = 2 * np.pi * k * lt / PM1
        W_E[t, OFFSET_LA + 2*ki] = np.cos(theta)
        W_E[t, OFFSET_LA + 2*ki + 1] = np.sin(theta)
# Tokens 0, 97 (op), 98 (eq): all zero (already done)

POS_EMB = np.zeros((4, D_MODEL), dtype=np.float32)
for i in range(4):
    POS_EMB[i, OFFSET_POS + i] = POS_MARKER_VAL

# Sanity: W_E[a=g] should have cos(2π·k·1/96), sin(2π·k·1/96) at positions 2k-2, 2k-1
print('W_E sanity check:')
print(f'  W_E[token=1, k=1 cos] = {W_E[1, 0]:.4f}  (expected cos(0) = 1.0 since dlog[1]=0)')
print(f'  W_E[token=g, k=1 cos] = {W_E[g, 0]:.4f}  (expected cos(2π/96) = {np.cos(2*np.pi/96):.4f})')
print(f'  W_E[token=g, k=1 sin] = {W_E[g, 1]:.4f}  (expected sin(2π/96) = {np.sin(2*np.pi/96):.4f})')
print(f'  W_E shape: {W_E.shape}, norm of W_E[1..96]: {np.linalg.norm(W_E[1:P], axis=1).mean():.3f}')
print(f'  POS_EMB diag: {[POS_EMB[i, OFFSET_POS+i] for i in range(4)]}')

In [ ]:
# Cell 4 — Build block 0 attention
# Heads:
#   Head 0: la transport. At eq (pos 3) attend to pos 0, copy Fourier 0..95 → eq's 0..95.
#   Head 1: lb transport. At eq attend to pos 2, copy Fourier 0..95 → eq's 96..191.
#   Heads 2, 3: zero (unused).
#
# nn.MultiheadAttention: in_proj_weight [3*d, d], rows split as [W_q | W_k | W_v].
# Per head h: W_q^h = in_proj_weight[h*head_dim : (h+1)*head_dim, :], etc.
# out_proj.weight [d, d] mixes head outputs (each head produces head_dim features).

in_proj_weight = np.zeros((3 * D_MODEL, D_MODEL), dtype=np.float32)
out_proj_weight = np.zeros((D_MODEL, D_MODEL), dtype=np.float32)

# After norm1, position markers (originally at OFFSET_POS+i, value POS_MARKER_VAL)
# survive RMSNorm with scale 1. Pure pos vectors (op, eq tokens) have RMS = POS_MARKER_VAL/sqrt(d).
# Fourier tokens (a, b) have RMS ≈ sqrt((K*2 * 0.5 + POS_MARKER_VAL^2)/d).
# Either way, post-norm magnitude at dim OFFSET_POS+i is POS_MARKER_VAL / RMS(input).
# We don't need exact magnitude — just that Q·K matches via the marker dims.

# --- Head 0: la transport, attends pos 0 from eq ---
# W_q^0: read dim OFFSET_POS+3 (eq's marker), output a direction in head_dim.
# W_k^0: read dim OFFSET_POS+0 (pos 0 marker), output the SAME direction.
# Use direction = e_0 in head_dim space (just put 1 at the 0-th component of head's output).
h = 0
Q_offset = h * HEAD_DIM
K_offset = D_MODEL + h * HEAD_DIM
V_offset = 2 * D_MODEL + h * HEAD_DIM
in_proj_weight[Q_offset + 0, OFFSET_POS + 3] = 1.0  # eq marker -> head out 0
in_proj_weight[K_offset + 0, OFFSET_POS + 0] = 1.0  # pos 0 marker -> head out 0
# W_v^0: read Fourier of la from dims OFFSET_LA..OFFSET_LA+95, output to head's 96 dims (identity)
for j in range(96):
    in_proj_weight[V_offset + j, OFFSET_LA + j] = 1.0
# out_proj for head 0: place head 0's output (96 dims) into d_model dims 0..95 (OFFSET_LA range)
for j in range(96):
    out_proj_weight[OFFSET_LA + j, h * HEAD_DIM + j] = 1.0

# --- Head 1: lb transport, attends pos 2 from eq ---
h = 1
Q_offset = h * HEAD_DIM
K_offset = D_MODEL + h * HEAD_DIM
V_offset = 2 * D_MODEL + h * HEAD_DIM
in_proj_weight[Q_offset + 0, OFFSET_POS + 3] = 1.0  # eq marker
in_proj_weight[K_offset + 0, OFFSET_POS + 2] = 1.0  # pos 2 marker (b token)
for j in range(96):
    in_proj_weight[V_offset + j, OFFSET_LA + j] = 1.0  # read same Fourier dims (token's Fourier)
# out_proj for head 1: place into dims OFFSET_LB..OFFSET_LB+95
for j in range(96):
    out_proj_weight[OFFSET_LB + j, h * HEAD_DIM + j] = 1.0

# Heads 2, 3: leave as zero (they contribute zero to output)

# In-proj bias is zero by default in our setup (PyTorch MHA has bias by default though)
in_proj_bias = np.zeros((3 * D_MODEL,), dtype=np.float32)
out_proj_bias = np.zeros((D_MODEL,), dtype=np.float32)

print('Attention weights built:')
print(f'  in_proj_weight: shape {in_proj_weight.shape}, nonzero entries: {(in_proj_weight != 0).sum()}')
print(f'  out_proj_weight: shape {out_proj_weight.shape}, nonzero entries: {(out_proj_weight != 0).sum()}')

In [ ]:
# Cell 5 — Build block 0 FFN (SwiGLU)
# For each k ∈ {1..48} we want to compute cos(k·(la-lb)) and sin(k·(la-lb)) and place
# them at OFFSET_COS+k-1 and OFFSET_SIN+k-1 in d_model output.
#
# Trigonometric identities:
#   cos(k·(la-lb)) =  cos(k·la)·cos(k·lb) + sin(k·la)·sin(k·lb)
#   sin(k·(la-lb)) =  sin(k·la)·cos(k·lb) - cos(k·la)·sin(k·lb)
#
# So for each k we need 4 mid-neurons (cos·cos, sin·sin, sin·cos, cos·sin) and combine via W2.
# Total active mid-neurons: 4 × 48 = 192 (out of 4·d_model = 1536 available).
#
# SwiGLU pure-product trick:
#   W1[i, :] = EPS_SWIGLU * (selector for la-Fourier component)
#   W3[i, :] = (selector for lb-Fourier component)
#   gate_i = SiLU(EPS_SWIGLU * la_component) ≈ EPS_SWIGLU * la_component / 2  (for small EPS)
#   gate_i * (W3 x)_i ≈ (EPS_SWIGLU / 2) * la_component * lb_component
#   W2 multiplies by GAIN_W2 = 2/EPS_SWIGLU to recover pure product.
#
# Mid-neuron index layout: for k_index ki ∈ {0..47}, k = ki+1:
#   neuron 4*ki + 0: la_cos · lb_cos  -> contributes +1 to cos((la-lb))_k
#   neuron 4*ki + 1: la_sin · lb_sin  -> contributes +1 to cos((la-lb))_k
#   neuron 4*ki + 2: la_sin · lb_cos  -> contributes +1 to sin((la-lb))_k
#   neuron 4*ki + 3: la_cos · lb_sin  -> contributes -1 to sin((la-lb))_k

D_MID = 4 * D_MODEL  # 1536
W1 = np.zeros((D_MID, D_MODEL), dtype=np.float32)
W3 = np.zeros((D_MID, D_MODEL), dtype=np.float32)
W2 = np.zeros((D_MODEL, D_MID), dtype=np.float32)

for ki, k in enumerate(K_FREQS):
    la_cos_dim = OFFSET_LA + 2*ki     # dim of cos(k·la) in d_model
    la_sin_dim = OFFSET_LA + 2*ki + 1
    lb_cos_dim = OFFSET_LB + 2*ki
    lb_sin_dim = OFFSET_LB + 2*ki + 1
    cos_out_dim = OFFSET_COS + ki
    sin_out_dim = OFFSET_SIN + ki

    # neuron 4ki + 0: la_cos · lb_cos -> +cos((la-lb))_k
    n = 4*ki + 0
    W1[n, la_cos_dim] = EPS_SWIGLU
    W3[n, lb_cos_dim] = 1.0
    W2[cos_out_dim, n] = GAIN_W2

    # neuron 4ki + 1: la_sin · lb_sin -> +cos((la-lb))_k
    n = 4*ki + 1
    W1[n, la_sin_dim] = EPS_SWIGLU
    W3[n, lb_sin_dim] = 1.0
    W2[cos_out_dim, n] = GAIN_W2

    # neuron 4ki + 2: la_sin · lb_cos -> +sin((la-lb))_k
    n = 4*ki + 2
    W1[n, la_sin_dim] = EPS_SWIGLU
    W3[n, lb_cos_dim] = 1.0
    W2[sin_out_dim, n] = GAIN_W2

    # neuron 4ki + 3: la_cos · lb_sin -> -sin((la-lb))_k
    n = 4*ki + 3
    W1[n, la_cos_dim] = EPS_SWIGLU
    W3[n, lb_sin_dim] = 1.0
    W2[sin_out_dim, n] = -GAIN_W2

print(f'FFN weights built:')
print(f'  W1: nonzero {(W1 != 0).sum()} (expected {4*K} = {4*K})')
print(f'  W3: nonzero {(W3 != 0).sum()}')
print(f'  W2: nonzero {(W2 != 0).sum()}')
print(f'  Active mid-neurons: 0..{4*K-1} (= {4*K})')

In [ ]:
# Cell 6 — Build head (W_U)
# After block 0, eq position residual has cos((la-lb))_k at OFFSET_COS+k-1 and sin at OFFSET_SIN+k-1.
# (The la/lb Fourier at 0..191 also remains in the residual stream — but we won't read those.)
#
# Logit_c should peak at c such that lc = la-lb mod 96.
# Set W_U[c, OFFSET_COS+k-1] = cos(2π·k·dlog(c)/96)
# Set W_U[c, OFFSET_SIN+k-1] = sin(2π·k·dlog(c)/96)
# Then logit_c = Σ_k [cos(2πk·lc/96)·cos(2πk·(la-lb)/96) + sin·sin] = Σ_k cos(2πk·(lc-(la-lb))/96).
# This sums to K=48 when lc = la-lb (correct), small otherwise — argmax picks correct c.

W_U = np.zeros((P, D_MODEL), dtype=np.float32)
for c in range(1, P):
    lc = dlog[c]
    for ki, k in enumerate(K_FREQS):
        theta = 2 * np.pi * k * lc / PM1
        W_U[c, OFFSET_COS + ki] = np.cos(theta)
        W_U[c, OFFSET_SIN + ki] = np.sin(theta)
# W_U[0, :] = 0 (never predict 0)

print(f'W_U built: shape {W_U.shape}, nonzero entries: {(W_U != 0).sum()}')

# norm_f scale: leave at 1. We rely on argmax being invariant to per-vector scaling.
norm_f_scale = np.ones(D_MODEL, dtype=np.float32)

In [ ]:
# Cell 7 — Block 1: make it a pure identity (zero attn output, zero FFN output)
# attention out = 0: set in_proj_weight = 0 and out_proj_weight = 0. Then attn(h, h, h) returns 0.
# FFN out = 0: set W2 = 0 (regardless of W1, W3).

in_proj_weight_b1 = np.zeros((3 * D_MODEL, D_MODEL), dtype=np.float32)
out_proj_weight_b1 = np.zeros((D_MODEL, D_MODEL), dtype=np.float32)
in_proj_bias_b1 = np.zeros((3 * D_MODEL,), dtype=np.float32)
out_proj_bias_b1 = np.zeros((D_MODEL,), dtype=np.float32)
W1_b1 = np.zeros((D_MID, D_MODEL), dtype=np.float32)
W2_b1 = np.zeros((D_MODEL, D_MID), dtype=np.float32)
W3_b1 = np.zeros((D_MID, D_MODEL), dtype=np.float32)
norm1_b1_scale = np.ones(D_MODEL, dtype=np.float32)
norm2_b1_scale = np.ones(D_MODEL, dtype=np.float32)

norm1_b0_scale = np.ones(D_MODEL, dtype=np.float32)
norm2_b0_scale = np.ones(D_MODEL, dtype=np.float32)

# --- Assemble state dict ---
def to_tensor(arr):
    return torch.from_numpy(arr.astype(np.float32))

state_dict = {
    'tok_emb.weight': to_tensor(W_E),
    'pos_emb.weight': to_tensor(POS_EMB),
    'blocks.0.norm1.scale': to_tensor(norm1_b0_scale),
    'blocks.0.attn.in_proj_weight': to_tensor(in_proj_weight),
    'blocks.0.attn.in_proj_bias': to_tensor(in_proj_bias),
    'blocks.0.attn.out_proj.weight': to_tensor(out_proj_weight),
    'blocks.0.attn.out_proj.bias': to_tensor(out_proj_bias),
    'blocks.0.norm2.scale': to_tensor(norm2_b0_scale),
    'blocks.0.w1.weight': to_tensor(W1),
    'blocks.0.w2.weight': to_tensor(W2),
    'blocks.0.w3.weight': to_tensor(W3),
    'blocks.1.norm1.scale': to_tensor(norm1_b1_scale),
    'blocks.1.attn.in_proj_weight': to_tensor(in_proj_weight_b1),
    'blocks.1.attn.in_proj_bias': to_tensor(in_proj_bias_b1),
    'blocks.1.attn.out_proj.weight': to_tensor(out_proj_weight_b1),
    'blocks.1.attn.out_proj.bias': to_tensor(out_proj_bias_b1),
    'blocks.1.norm2.scale': to_tensor(norm2_b1_scale),
    'blocks.1.w1.weight': to_tensor(W1_b1),
    'blocks.1.w2.weight': to_tensor(W2_b1),
    'blocks.1.w3.weight': to_tensor(W3_b1),
    'norm_f.scale': to_tensor(norm_f_scale),
    'head.weight': to_tensor(W_U),
}

missing, unexpected = m.load_state_dict(state_dict, strict=False)
print(f'load_state_dict: missing={missing}, unexpected={unexpected}')
m.eval()
print('Handcrafted state_dict loaded into model.')

In [ ]:
# Cell 8 — Diagnostic: per-stage activations on a few examples
# Goal: verify each engineered stage produces the expected pattern.

@torch.no_grad()
def trace_model(a_val, b_val):
    a_t = torch.tensor([a_val], device=device)
    b_t = torch.tensor([b_val], device=device)
    B = 1
    op = torch.full((B,), P, device=device, dtype=torch.long)
    eq = torch.full((B,), P+1, device=device, dtype=torch.long)
    tok = torch.stack([a_t, op, b_t, eq], dim=1)
    pos = torch.arange(4, device=device).unsqueeze(0).expand(B, -1)
    x = m.tok_emb(tok) + m.pos_emb(pos)  # [B=1, 4, 384]
    # After tok+pos
    print(f'  After tok+pos (eq pos): nonzero dims {((x[0,3] != 0).nonzero().squeeze(-1)).tolist()[:10]}...')
    # Block 0 attn
    h = m.blocks[0].norm1(x)
    attn_out, _ = m.blocks[0].attn(h, h, h, need_weights=False)
    x1 = x + attn_out
    print(f'  After attn at eq: la-Fourier 0..95 norm = {x1[0,3, OFFSET_LA:OFFSET_LA+96].norm().item():.3f}, '
          f'lb-Fourier 96..191 norm = {x1[0,3, OFFSET_LB:OFFSET_LB+96].norm().item():.3f}')
    # Block 0 FFN
    h2 = m.blocks[0].norm2(x1)
    gate = F.silu(m.blocks[0].w1(h2))
    up = m.blocks[0].w3(h2)
    ffn_mid = gate * up
    ffn_out = m.blocks[0].w2(ffn_mid)
    x2 = x1 + ffn_out
    print(f'  After FFN at eq: cos((la-lb)) dims 192..239 norm = {x2[0,3, OFFSET_COS:OFFSET_COS+K].norm().item():.3f}, '
          f'sin dims 240..287 norm = {x2[0,3, OFFSET_SIN:OFFSET_SIN+K].norm().item():.3f}')
    # Block 1 (should be identity)
    x3 = m.blocks[1](x2)
    diff_b1 = (x3 - x2)[0, 3].norm().item()
    print(f'  After block 1: residual change at eq = {diff_b1:.6f} (should be ≈0)')
    # Head
    hf = m.norm_f(x3)[:, -1, :]
    logits = m.head(hf)
    pred = int(logits.argmax(-1).item())
    expected_c = (a_val * pow(b_val, P-2, P)) % P
    print(f'  prediction: {pred}, expected: {expected_c}, match: {pred == expected_c}')
    print(f'  top-3 logits: {[(int(i.item()), float(logits[0, i].item())) for i in logits[0].topk(3).indices]}')
    # Also: check cos((la-lb)) values match analytical prediction
    la_v = int(dlog[a_val]); lb_v = int(dlog[b_val])
    print(f'  la={la_v}, lb={lb_v}, la-lb mod 96 = {(la_v - lb_v) % PM1}')
    for ki, k in enumerate(K_FREQS[:3]):
        theta = 2 * np.pi * k * (la_v - lb_v) / PM1
        expected_cos = np.cos(theta)
        observed_cos = x2[0, 3, OFFSET_COS + ki].item()
        print(f'    k={k}: expected cos = {expected_cos:.4f}, observed = {observed_cos:.4f}, ratio = {observed_cos/expected_cos if abs(expected_cos)>1e-6 else float("nan"):.3f}')
    return pred, expected_c

print('=== Trace (a=3, b=5) ===')
trace_model(3, 5)
print('\n=== Trace (a=10, b=7) ===')
trace_model(10, 7)

In [ ]:
# Cell 9 — Full-dataset accuracy
@torch.no_grad()
def full_eval(model):
    BS = 512
    correct = 0; total = 0
    per_class_correct = np.zeros(PM1)
    per_class_total = np.zeros(PM1)
    for i in range(0, len(all_a_t), BS):
        a = all_a_t[i:i+BS]; b = all_b_t[i:i+BS]; c = all_c_t[i:i+BS]
        logits = model(a, b)
        pred = logits.argmax(-1)
        match = (pred == c).cpu().numpy()
        correct += int(match.sum()); total += len(a)
        # bucket by (la-lb) mod 96
        for j in range(len(a)):
            la = int(dlog[int(a[j].item())])
            lb = int(dlog[int(b[j].item())])
            diff = (la - lb) % PM1
            per_class_correct[diff] += match[j]
            per_class_total[diff] += 1
    return correct/total, per_class_correct, per_class_total

t0 = time.time()
acc, pcc, pct = full_eval(m)
print(f'Handcrafted full-dataset accuracy: {acc*100:.2f}%   ({time.time()-t0:.1f}s)')
print(f'Per-(la-lb) accuracy distribution:')
rates = pcc / np.maximum(pct, 1)
print(f'  worst 5: {sorted(enumerate(rates), key=lambda x: x[1])[:5]}')
print(f'  best 5:  {sorted(enumerate(rates), key=lambda x: x[1])[-5:]}')

In [ ]:
# Cell 10 — Mechanistic diagnostics: W_U Fourier circle CV, residual manifold cleanness
@torch.no_grad()
def residual_manifold(model):
    out = np.zeros((PM1, D_MODEL)); counts = np.zeros(PM1)
    BS = 1024
    for i in range(0, len(all_a_t), BS):
        a = all_a_t[i:i+BS]; b = all_b_t[i:i+BS]
        _, h = model.forward_capture_pre_head(a, b)
        h_np = h.cpu().numpy()
        for j in range(len(a)):
            la = int(dlog[int(a[j].item())])
            lb = int(dlog[int(b[j].item())])
            di = (la - lb) % PM1
            out[di] += h_np[j]; counts[di] += 1
    out /= np.maximum(counts[:, None], 1)
    return out

def circle_cv(M, k):
    p = M.shape[0]; i = np.arange(p)
    Mc = M - M.mean(axis=0)
    A = (np.cos(2*np.pi*k*i/p) @ Mc) / (p/2)
    B = (np.sin(2*np.pi*k*i/p) @ Mc) / (p/2)
    u = A / (np.linalg.norm(A) + 1e-12)
    v = B - (B @ u) * u; v = v / (np.linalg.norm(v) + 1e-12)
    xy = np.stack([Mc @ u, Mc @ v], axis=1)
    cx, cy = xy.mean(0)
    r = np.sqrt((xy[:,0]-cx)**2 + (xy[:,1]-cy)**2)
    return r.std() / (r.mean() + 1e-12), xy

manifold = residual_manifold(m)
print(f'Residual manifold shape: {manifold.shape}, mean row norm: {np.linalg.norm(manifold, axis=1).mean():.2f}')

# W_U circle CV in dlog order (over 96 tokens c indexed by dlog)
head_w = state_dict['head.weight'].numpy()  # [P, D_MODEL]
head_dlog = head_w[exp_table]  # [96, D_MODEL] — order by dlog

print('\nCircle CV (W_U, residual manifold) for selected k:')
for k in [1, 11, 19, 25, 35, 44, 48]:
    cv_wu, _ = circle_cv(head_dlog, k)
    cv_mf, _ = circle_cv(manifold, k)
    print(f'  k={k:2d}: W_U CV = {cv_wu:.3f}, manifold CV = {cv_mf:.3f}  (expected: ~0.00 for both — perfect circles)')

In [ ]:
# Cell 11 — Visualize: W_U dlog circle at k=11, residual manifold at k=11, FFN1-mid sample neurons
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))

ax = axes[0]
_, xy_wu = circle_cv(head_dlog, 11)
ax.scatter(xy_wu[:, 0], xy_wu[:, 1], s=20, c=range(96), cmap='hsv', alpha=0.9)
ax.set_aspect('equal')
ax.set_title(f'Handcrafted W_U at k=11 in dlog basis')
ax.set_xlabel('cos plane'); ax.set_ylabel('sin plane')

ax = axes[1]
_, xy_mf = circle_cv(manifold, 11)
ax.scatter(xy_mf[:, 0], xy_mf[:, 1], s=20, c=range(96), cmap='hsv', alpha=0.9)
ax.set_aspect('equal')
ax.set_title(f'Handcrafted residual manifold at k=11')
ax.set_xlabel('cos plane'); ax.set_ylabel('sin plane')

# FFN1-mid activations averaged per (la-lb): pick the 4 neurons for k=11 we engineered
ki_target = K_FREQS.index(11)
neuron_ids = [4*ki_target + j for j in range(4)]
labels_ffn = ['cos·cos (→+cos)', 'sin·sin (→+cos)', 'sin·cos (→+sin)', 'cos·sin (→-sin)']
ax = axes[2]
ffn_means = np.zeros((4, PM1))
ffn_counts = np.zeros(PM1)
BS = 1024
with torch.no_grad():
    for i in range(0, len(all_a_t), BS):
        a = all_a_t[i:i+BS]; b = all_b_t[i:i+BS]
        B = a.size(0)
        op = torch.full((B,), P, device=device, dtype=torch.long)
        eq = torch.full((B,), P+1, device=device, dtype=torch.long)
        tok = torch.stack([a, op, b, eq], dim=1)
        pos = torch.arange(4, device=device).unsqueeze(0).expand(B, -1)
        x = m.tok_emb(tok) + m.pos_emb(pos)
        h = m.blocks[0].norm1(x); attn_out, _ = m.blocks[0].attn(h, h, h, need_weights=False)
        x1 = x + attn_out
        h2 = m.blocks[0].norm2(x1)
        gate = F.silu(m.blocks[0].w1(h2)); up = m.blocks[0].w3(h2); mid = (gate * up)[:, -1, :]
        mid_np = mid.cpu().numpy()
        for j in range(B):
            la = int(dlog[int(a[j].item())]); lb = int(dlog[int(b[j].item())])
            di = (la - lb) % PM1
            for ni, nid in enumerate(neuron_ids):
                ffn_means[ni, di] += mid_np[j, nid]
            ffn_counts[di] += 1
ffn_means /= np.maximum(ffn_counts[None, :], 1)
for ni in range(4):
    ax.plot(ffn_means[ni], lw=1.5, alpha=0.8, label=labels_ffn[ni])
ax.set_xlabel('la − lb mod 96')
ax.set_ylabel('FFN1-mid activation')
ax.set_title(f'FFN1-mid neurons for k=11')
ax.legend(fontsize=7, frameon=False, loc='best')

plt.tight_layout()
plt.savefig(OUT_DIR / 'figures.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 12 — Save state_dict + summary
torch.save(state_dict, OUT_DIR / 'handcrafted_state_dict.pt')

summary = {
    'config': {
        'P': P, 'PM1': PM1, 'g': int(g),
        'd_model': D_MODEL, 'n_heads': N_HEADS, 'head_dim': HEAD_DIM,
        'n_blocks_active': 1, 'k_freqs': K_FREQS, 'K': K,
        'eps_swiglu': EPS_SWIGLU, 'gain_w2': GAIN_W2, 'pos_marker_val': POS_MARKER_VAL,
        'layout': {
            'OFFSET_LA': OFFSET_LA, 'OFFSET_LB': OFFSET_LB,
            'OFFSET_COS': OFFSET_COS, 'OFFSET_SIN': OFFSET_SIN, 'OFFSET_POS': OFFSET_POS,
        },
    },
    'metrics': {
        'full_accuracy': float(acc),
        'per_diff_accuracy': rates.tolist(),
        'wu_circle_cv': {int(k): float(circle_cv(head_dlog, k)[0]) for k in [1, 11, 19, 25, 35, 44, 48]},
        'manifold_circle_cv': {int(k): float(circle_cv(manifold, k)[0]) for k in [1, 11, 19, 25, 35, 44, 48]},
    },
}
with open(OUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved: {OUT_DIR / "handcrafted_state_dict.pt"}')
print(f'Saved: {OUT_DIR / "summary.json"}')
print(f'Saved: {OUT_DIR / "figures.png"}')
print(f'\n=== Final accuracy: {acc*100:.2f}% ===')

## What this verifies

If the handcrafted model achieves ≥ 95% full-dataset accuracy, every mechanistic claim from the trained-model chronicle is *constructively* verified:

- **W_E carries dlog Fourier of token id.** (We literally wrote it that way; the algorithm uses these features as inputs.)
- **Attention V projects copy Fourier features of `a` and `b` to the eq position** (Plan 4 finding).
- **FFN1-mid neurons compute single-frequency `cos(k·(la−lb))`** via the product identity (Plan 4 finding).
- **W_U reads from these Fourier channels, with per-c weight = `cos(2πk·lc/96)`** (α finding).
- **The residual manifold at pre_head has clean Fourier circles** (Plan 1f finding) — because we explicitly wrote the cos/sin channels into the residual stream.

## What might fail on first try

- **SwiGLU non-linearity.** `SiLU(eps·x) ≈ eps·x/2` only to first order. For `eps = 0.01` and inputs in `[-1, 1]`, the residual error is `O(eps²) ≈ 10⁻⁴` per neuron — tiny. If accuracy drops below 95%, decrease `EPS_SWIGLU` (more linear, less harmonic contamination) and recompute `GAIN_W2`.
- **Position marker survival through RMSNorm.** If markers get washed out (e.g., overwhelmed by Fourier signal), attention won't route to correct positions. Diagnose by checking `Cell 8` traces — `la-Fourier norm` and `lb-Fourier norm` should both be `≈ 1` after attn. If 0, the attention isn't routing.
- **RMSNorm rescaling at norm_f.** The post-FFN residual has cos/sin channels at scale ≈ 1, but the la/lb Fourier from attention is also still present (norm ≈ 1 each). RMSNorm normalizes the whole vector, so each channel's effective amplitude depends on the total RMS. Could cause logit-scale issues but shouldn't affect argmax.
- **Two-peak ambiguity.** If we only used cos channels (no sin), W_U would peak at both `c` and `c⁻¹ mod 97` (the reciprocal). Using both cos and sin breaks the symmetry — this is why we engineered 4 neurons per `k`, not 2.